# On-Device Voice Agent — Local Jupyter Runner (WSL)

Runs the **RQ2 latency-split** experiment on *your* CPU — the most faithful
setting for this paper (commodity CPU, not a shared cloud vCPU).

The code is already on disk, so there's no clone step. We just: point the kernel
at the repo root → install runtimes → download one small model → dummy sanity run
→ real RQ2 → plot.

**Detected:** 8 cores, ~7.7 GB RAM → use the **3B Q4_K_M** model (a 7B would be
tight). Close other apps before the real run to free RAM.

## 1. Point the kernel at the repo root
Jupyter starts in `notebooks/`; we `chdir` up so `import src...` and `!python -m`
both resolve.

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
print('cwd:', os.getcwd())
assert os.path.isdir('src'), 'run this from the project root (should see src/)'
print('ok — repo root found')

## 2. Install runtimes
`llama-cpp-python` compiles a CPU wheel (~1-3 min on 8 cores). Run once per venv.

In [ ]:
!pip install -q -r requirements.txt -r requirements-models.txt
print('deps installed')

## 3. Download one small model (~2 GB) + Piper voice
Cached under `models/`. faster-whisper downloads its `base` model on first use.

In [ ]:
import os
from huggingface_hub import hf_hub_download
os.makedirs('models', exist_ok=True)

llm_path = hf_hub_download(
    repo_id='bartowski/Qwen2.5-3B-Instruct-GGUF',
    filename='Qwen2.5-3B-Instruct-Q4_K_M.gguf', local_dir='models')
print('LLM :', llm_path)

voice = hf_hub_download(repo_id='rhasspy/piper-voices',
    filename='en/en_US/amy/medium/en_US-amy-medium.onnx', local_dir='models')
hf_hub_download(repo_id='rhasspy/piper-voices',
    filename='en/en_US/amy/medium/en_US-amy-medium.onnx.json', local_dir='models')
os.environ['VA_PIPER_VOICE'] = voice
os.environ['VA_ASR_MODEL'] = 'base'      # small/fast on CPU
os.environ['VA_ASR_COMPUTE'] = 'int8'
print('voice:', voice)

## 4. Sanity check — dummy run (instant, no models)
Call the experiment's `run()` directly so you can inspect objects in-notebook.

In [ ]:
from importlib import reload
import src.experiments.rq2_latency as rq2
reload(rq2)
rq2.run(dummy=True, quant='dummy', model_path=None, n_turns=3)

## 5. Real RQ2 on your CPU
First call loads the model + downloads whisper `base` (~1 min one-time). Then it
runs 5 turns and prints the per-stage breakdown + dominance verdict.

In [ ]:
import src.experiments.rq2_latency as rq2
reload(rq2)
rq2.run(dummy=False, quant='Q4_K_M',
        model_path='models/Qwen2.5-3B-Instruct-Q4_K_M.gguf', n_turns=5)

## 6. Plot the latency split — your draft Figure 1

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, glob
f = sorted(glob.glob('results/rq2_latency_*.jsonl'))[-1]
df = pd.read_json(f, lines=True)
m = df[['asr_s','llm_decode_s','tts_total_s']].mean() * 1000
ax = m.plot.bar(color=['#4C72B0','#C44E52','#55A868'], figsize=(5,3.5))
ax.set_ylabel('mean latency (ms)'); ax.set_title('RQ2 latency split (CPU, Q4_K_M)')
for i,v in enumerate(m): ax.text(i, v, f'{v:.0f}', ha='center', va='bottom')
plt.tight_layout(); plt.show()
dom = m.idxmax()
print(f'Dominant: {dom} ({m[dom]/m.sum()*100:.0f}% of stage time)')
print('LLM-dominated → confirms hypothesis, contra refuted "TTS dominates".'
      if dom=='llm_decode_s' else 'Surprise — investigate.')

## Next
- **Quant sweep:** download `Q8_0, Q5_K_M, Q3_K_M, Q2_K` of the 3B and loop the
  cell-5 call over each `--model-path` → latency-vs-quant curve.
- **Bigger model:** if RAM allows, try a 7B Q4_K_M to match the paper target and
  show the LLM share grows vs the H100 baseline.
- Then RQ1 (`rq1_quality`) once BFCL/MT-Bench data is wired.